# DevGraph-RL — RLHF Training (Colab)

**Before running:** Runtime → Change runtime type → **T4 GPU**

Run cells top to bottom. After Cell 1, restart runtime before continuing.

## Cell 0 — GPU Check

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    raise RuntimeError('No GPU. Go to Runtime > Change runtime type > T4 GPU')

## Cell 1 — Install Dependencies

After this cell finishes: **Runtime → Restart runtime**, then continue from Cell 2.

In [ ]:
import subprocess, sys
for pkg in ['torchao>=0.16.0','trl>=0.9.0','peft>=0.11.0','transformers>=4.41.0','datasets>=2.20.0','scikit-learn>=1.5.0','huggingface_hub>=0.23.0','keras>=3.3.0','jax','jaxlib']:
    print(f'Installing {pkg}...')
    subprocess.run([sys.executable,'-m','pip','install','-q',pkg],check=True)
print('\nDone. NOW: Runtime > Restart runtime, then run from Cell 2.')

## Cell 2 — Config

Edit the values below.

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'jax'

GITHUB_REPO       = 'https://github.com/YOUR_USERNAME/devgraph-rl.git'
HF_TOKEN          = ''                   # paste HF token to push model
HF_MODEL_ID       = ''                   # e.g. 'yourname/devgraph-rl-v1'
BASE_MODEL        = 'Qwen/Qwen2.5-0.5B'
N_TRAIN_EPOCHS    = 3
N_SWEEP_EPOCHS    = 3
MAX_SWEEP_CONFIGS = 4
MIN_SCORE         = 0.0

print('Config set.')
print(f'  Base model  : {BASE_MODEL}')
print(f'  Train epochs: {N_TRAIN_EPOCHS}')
print(f'  Push to Hub : {bool(HF_TOKEN and HF_MODEL_ID)}')

## Cell 3 — Clone Repo and Patch for trl 1.5.1

In [ ]:
import sys, os, re
from pathlib import Path

if not Path('/content/devgraph-rl').exists():
    os.system(f'git clone {GITHUB_REPO} /content/devgraph-rl')
else:
    os.system('cd /content/devgraph-rl && git pull')

os.chdir('/content/devgraph-rl')
sys.path.insert(0, '/content/devgraph-rl')
print(f'Working directory: {os.getcwd()}')

# Patch torch_trainer.py for trl 1.5.1
p = Path('/content/devgraph-rl/src/training/torch_trainer.py')
c = p.read_text()

# 1. Fix torch_dtype deprecation
c = c.replace('torch_dtype=dtype', 'dtype=dtype')

# 2. Remove max_prompt_length entirely
c = re.sub(r'[ \t]*max_prompt_length=self\.config\.max_prompt_length,?\n', '', c)

# 3. In DPOConfig block: ensure max_length present, remove from DPOTrainer
# Remove max_length from DPOTrainer call if present
c = re.sub(r'[ \t]*max_length=self\.config\.max_length,?\n([ \t]*processing_class)', r'\1', c)

# 4. Remove tokenizer= from DPOTrainer (old API)
c = re.sub(r'[ \t]*tokenizer=tokenizer,?\n', '', c)

# 5. Ensure max_length is in DPOConfig after learning_rate
if 'max_length=self.config.max_length' not in c.split('def _build_dpo_trainer')[1].split('def _run_training')[0]:
    c = c.replace(
        'learning_rate=self.config.learning_rate,\n            logging_steps',
        'learning_rate=self.config.learning_rate,\n            max_length=self.config.max_length,\n            logging_steps'
    )

p.write_text(c)
print('torch_trainer.py patched for trl', __import__('trl').__version__)

## Cell 4 — Upload Reward History

Copy from WSL2 first:
```bash
cp ~/devgraph-rl/data/vector_store/reward_history.jsonl /mnt/c/Users/YOUR_USERNAME/Downloads/
```

In [ ]:
from google.colab import files
import shutil, json
from pathlib import Path

dest_dir = Path('/content/devgraph-rl/data/vector_store')
dest_dir.mkdir(parents=True, exist_ok=True)

print('Select your reward_history.jsonl file:')
uploaded = files.upload()

for fname in uploaded:
    dest = dest_dir / 'reward_history.jsonl'
    shutil.move(fname, dest)
    records = [json.loads(l) for l in dest.read_text().strip().splitlines() if l.strip()]
    print(f'Uploaded {len(records)} records')
    for r in records[:3]:
        print(f"  score={r.get('final_score',0):.3f} | task={r.get('task','')[:50]}")

## Cell 5 — Sklearn Analysis

In [ ]:
from src.rewards.reward_store import get_reward_store
from src.training.data_collector import DataCollector
from src.training.sklearn_analyzer import SklearnAnalyzer

store    = get_reward_store()
analyzer = SklearnAnalyzer(min_score_delta=0.15)
collector= DataCollector(reward_store=store, analyzer=analyzer, min_score=MIN_SCORE)
analyzed = collector.collect_and_analyze(min_score=MIN_SCORE)

dist = analyzed.distribution
print('=== Score Distribution ===')
print(f'  Total  : {dist.total}')
print(f'  High   : {dist.high}  (>= 0.75)')
print(f'  Medium : {dist.medium}')
print(f'  Low    : {dist.low}  (< 0.40)')
print(f'  Mean   : {dist.mean:.3f} +/- {dist.std:.3f}')
print(f'\n=== Training Pairs ===')
print(f'  Selected: {analyzed.pairs.pairs_selected}')
for p in analyzed.pairs.pairs:
    print(f'  {p.task!r:35} delta={p.score_delta:.3f}')
print(f'\nReady: {analyzed.ready_for_training}')
assert analyzed.ready_for_training, 'Need >= 5 pairs. Score more outputs in the Rewards tab.'

## Cell 6 — Build Dataset

In [ ]:
from src.training.dataset_builder import DatasetBuilder

builder = DatasetBuilder()
split   = builder.build_split(analyzed, eval_ratio=0.15)
print(f'Train: {split.n_train} | Eval: {split.n_eval} | Delta: {split.mean_delta:.3f}')
print(f'Columns: {split.dataset_dict["train"].column_names}')

## Cell 7 — Keras Sweep (optional)

In [ ]:
import hashlib, struct
from pathlib import Path
from src.training.keras_experiments import KerasExperiments

def text_to_vec(text, dim=384):
    vec=[]
    seed=text.encode()
    for i in range(dim):
        h=hashlib.md5(seed+i.to_bytes(4,'little')).digest()
        vec.append(struct.unpack('f',h[:4])[0])
    norm=sum(x*x for x in vec)**0.5 or 1.0
    return [x/norm for x in vec]

train_emb=[{'chosen_embedding':text_to_vec(p.chosen_output),'rejected_embedding':text_to_vec(p.rejected_output)} for p in analyzed.pairs.pairs]
eval_emb=train_emb[:max(1,len(train_emb)//10)]

ke=KerasExperiments(experiment_dir=Path('/content/devgraph-rl/data/experiments'),embedding_dim=384)
sweep_space={'learning_rate':[1e-3,2e-4],'batch_size':[8,16],'hidden_dim':[64,128],'dropout':[0.1]}

print(f'Starting sweep — {MAX_SWEEP_CONFIGS} configs x {N_SWEEP_EPOCHS} epochs...')
sweep=ke.run_sweep(train_data=train_emb,eval_data=eval_emb,sweep_space=sweep_space,n_epochs=N_SWEEP_EPOCHS,max_configs=MAX_SWEEP_CONFIGS)

print(f'Best loss : {sweep.best_eval_loss:.4f}')
print(f'Best lr   : {sweep.best_config.learning_rate}')
print(f'Best batch: {sweep.best_config.batch_size}')
for r in sweep.ranked_runs[:4]:
    print(f'  {r.config.experiment_id:8} eval={r.eval_loss:.4f} lr={r.config.learning_rate} b={r.config.batch_size}')

best_lr   =sweep.best_config.learning_rate if sweep.best_eval_loss<float('inf') else 2e-4
best_batch=sweep.best_config.batch_size    if sweep.best_eval_loss<float('inf') else 8
print(f'\nUsing lr={best_lr} batch={best_batch} for DPO.')

## Cell 8a — DPO Config

In [ ]:
import torch
from src.training.torch_trainer import TorchTrainer, TrainerConfig

config=TrainerConfig(
    base_model=BASE_MODEL,
    output_dir='/content/devgraph-rl/data/checkpoints',
    n_epochs=N_TRAIN_EPOCHS,
    learning_rate=best_lr,
    per_device_train_batch_size=best_batch,
    per_device_eval_batch_size=best_batch,
    push_to_hub=bool(HF_TOKEN and HF_MODEL_ID),
    hub_model_id=HF_MODEL_ID or None,
    hub_token=HF_TOKEN or None,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    lora_r=16,lora_alpha=32,max_length=512,
)
warns=TorchTrainer(config).validate_config()
[print(f'  Warning: {w}') for w in warns] if warns else print('Config OK')
print(f'LR={config.learning_rate} batch={config.per_device_train_batch_size} bf16={config.bf16}')

## Cell 8b — DPO Training (1-3 hrs on T4)

In [ ]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

trainer=TorchTrainer(config)
result =trainer.train(split.dataset_dict)

print(f'\n=== Training Complete ===')
print(f'  Success       : {result.success}')
print(f'  Best loss     : {result.best_loss:.4f}' if result.best_loss!=float('inf') else '  Best loss     : n/a')
print(f'  Final delta   : +{result.final_reward_delta:.3f}')
print(f'  Improved      : {result.improved}')
print(f'  Duration      : {result.total_duration_seconds/3600:.1f} hrs')
if result.error: print(f'  Error         : {result.error}')
if result.hub_model_id: print(f'  Hub           : https://huggingface.co/{result.hub_model_id}')
if result.epoch_metrics:
    print('\n  Epochs:')
    for m in result.epoch_metrics:
        print(f'    {m.epoch}: loss={m.loss:.4f} chosen={m.reward_chosen:.3f} rejected={m.reward_rejected:.3f} delta=+{m.reward_delta:.3f}')

## Cell 9 — Download Result

In [ ]:
from google.colab import files
from pathlib import Path

rp=Path('/content/devgraph-rl/data/checkpoints/training_result.json')
if rp.exists():
    files.download(str(rp))
    print('Downloaded training_result.json')
    print('Copy to: ~/devgraph-rl/data/checkpoints/training_result.json')
    print('Then click Load History in the ML Lab tab.')
else:
    print('No result file — training may have failed.')

## Troubleshooting

| Problem | Fix |
|---|---|
| `CUDA out of memory` | Set `best_batch = 2` and `config.lora_r = 8` in Cell 8a |
| `Not enough pairs` | Score more outputs in Rewards tab, re-upload jsonl |
| Keras sweep all inf | Ignored — Cell 8 uses default lr/batch |
| Colab disconnects | Re-run Cell 8b only — checkpoint saved every 100 steps |
| Any import error | Re-run Cell 1, then Runtime → Restart runtime |